In [15]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tanushkumaryadav/wellll/DXMag_HeuslerDB_2025-09-09.csv


In [16]:
!pip install matminer

In [17]:
from matminer.datasets import load_dataset

df = load_dataset("heusler_magnetic")
print(df.columns.tolist())

['formula', 'heusler type', 'num_electron', 'struct type', 'latt const', 'tetragonality', 'e_form', 'pol fermi', 'mu_b', 'mu_b saturation']


In [18]:
from matminer.datasets import load_dataset

# Load the dataset
df = load_dataset("heusler_magnetic")

# Filter using 'type' instead of 'heusler_type'
half_heusler_df = df[df['heusler type'] == 'Half Heusler']
heusler_df= df[df['heusler type'] == 'Full Heusler']
print(f"Found {len(half_heusler_df)} Half-Heusler materials!")
print(half_heusler_df.head())

Found 449 Half-Heusler materials!
    formula  heusler type  num_electron struct type  latt const  \
576  CrTiGa  Half Heusler            13         C1b        6.03   
577  CrTiIn  Half Heusler            13         C1b        6.32   
578  CrTiSi  Half Heusler            14         C1b        5.85   
579  CrTiGe  Half Heusler            14         C1b        5.95   
580  CrTiSn  Half Heusler            14         C1b        6.25   

     tetragonality  e_form  pol fermi    mu_b  mu_b saturation  
576         1.0000   0.401      18.18  4.9901           844.28  
577         1.0000   0.545     100.00  5.0000           734.76  
578         1.0000   0.189     100.00  4.0000           741.17  
579         1.0000   0.217     100.00  4.0000           704.43  
580         1.0016   0.274     100.00  4.0000           606.81  


In [19]:
half_heusler_df.head()

,formula,heusler type,num_electron,struct type,latt const,tetragonality,e_form,pol fermi,mu_b,mu_b saturation
576,CrTiGa,Half Heusler,13,C1b,6.03,1.0000,0.401,18.18,4.9901,844.28
577,CrTiIn,Half Heusler,13,C1b,6.32,1.0000,0.545,100.00,5.0000,734.76
578,CrTiSi,Half Heusler,14,C1b,5.85,1.0000,0.189,100.00,4.0000,741.17
579,CrTiGe,Half Heusler,14,C1b,5.95,1.0000,0.217,100.00,4.0000,704.43
580,CrTiSn,Half Heusler,14,C1b,6.25,1.0016,0.274,100.00,4.0000,606.81


**Full Heusler Dataset**

In [20]:
heusler_df.sample(10)

,formula,heusler type,num_electron,struct type,latt const,tetragonality,e_form,pol fermi,mu_b,mu_b saturation
217,Fe2ScGa,Full Heusler,22,L21,6.0100,1.0000,-0.140,93.61,1.9142,327.11
7,V2ScAs,Full Heusler,18,D022,5.3100,1.6742,-0.162,0.00,0.0002,0.03
371,Co2TiIn,Full Heusler,25,L21,6.0700,1.0000,-0.251,88.37,1.0284,170.58
52,V2FeAs,Full Heusler,23,D022,4.9572,1.6374,-0.235,25.94,1.3832,257.25
234,Fe2VAl,Full Heusler,24,L21,5.6925,1.0001,-0.427,0.00,0.0000,0.00
41,V2MnSn,Full Heusler,21,L21,6.3200,1.0000,0.071,48.97,1.4920,219.25
483,Rh2FeP,Full Heusler,31,D022,5.4800,1.2737,-0.271,43.22,3.4132,604.05
55,V2CoGa,Full Heusler,22,D022,4.9264,1.6825,-0.184,0.00,0.0042,0.77
488,Rh2CoIn,Full Heusler,30,L21,6.2300,1.0000,-0.078,90.25,2.9963,459.67
156,Mn2TiSi,Full Heusler,22,L21,5.7700,1.0000,-0.535,93.46,1.9832,382.97


In [21]:
heusler_df['struct type'].value_counts()

struct type
L21     340
D022    236
Name: count, dtype: int64

In [22]:
heusler_df.shape

(576, 10)

In [23]:
heusler_df=heusler_df[["formula"]]
heusler_df['choice']=1
heusler_df.sample(10)

,formula,choice
316,Ru2CrGa,1
24,V2VP,1
509,Ni2ScSn,1
511,Ni2ScAs,1
184,Mn2MnGe,1
48,V2FeSi,1
421,Co2CoAs,1
428,Co2NiSn,1
129,Cr2CoSi,1
176,Mn2CrSn,1


In [24]:
heusler_df[heusler_df['formula'].duplicated()]

,formula,choice


In [25]:
import requests
import pandas as pd

url = "https://aflowlib.org/API/aflux/?$nspecies(3),compound,aflow_prototype_label_relax,$paging(1,1500)"

data = requests.get(url).json()

In [26]:
from pymatgen.core import Composition
def check_ab2c(formula):
    comp=Composition(formula)
    if sorted(comp.values())==[1,1,2]:
        return True
    else:
        return False

In [27]:
formula = [entry["compound"] for entry in data if "compound" in entry]
print(formula[10])
print(check_ab2c(formula[10]))

Cu1Ta1Zn1
False


In [28]:
neg=[]
for m in data:
    if(check_ab2c(m['compound']) and m['aflow_prototype_label_relax']!="AB2C_cF16_225_a_c_b" and m['aflow_prototype_label_relax']!='AB2C_tI8_139_a_e_b'):
        neg.append(m['compound'])

In [29]:
neg=set(neg)

In [30]:
len(neg)

1149

In [31]:
df_neg=pd.DataFrame({"formula":list(neg)})

In [32]:
df_neg['choice']=0

In [33]:
df_final=pd.concat([heusler_df,df_neg])
df_final.sample(10)

,formula,choice
67,V2NiGe,1
202,Ir1Nb1Tc2,0
521,Ni2TiSb,1
629,Li1Pt2Sc1,0
929,Hg2Pb1Si1,0
378,Ge1Hf2Tc1,0
346,Ru2CoGe,1
619,Br2Os1Pt1,0
238,Fe2VGe,1
614,Br2Nb1Tc1,0


In [34]:
duplicates = df_final[df_final['formula'].duplicated(keep=False)]

print(duplicates)

Empty DataFrame
Columns: [formula, choice]
Index: []


In [35]:
df_final.to_csv('check_fh.csv', index=False)

**Half Heusler dataset**

In [36]:
half_heusler_df.sample(10)

,formula,heusler type,num_electron,struct type,latt const,tetragonality,e_form,pol fermi,mu_b,mu_b saturation
585,CrVGa,Half Heusler,14,C1b,5.8200,1.0000,0.527,25.64,0.0306,5.76
701,FeCrAl,Half Heusler,17,C1b,5.4800,1.0000,0.205,6.07,0.6708,151.21
760,RuCrSn,Half Heusler,18,C1b,5.9900,0.9983,0.246,0.00,0.0000,0.00
982,MnMnAl,Half Heusler,17,triclinic,-0.0204,-0.0146,NaN,NaN,0.3585,NaN
736,FeNiSb,Half Heusler,23,C1b,5.8300,1.0034,0.122,46.72,2.5198,470.11
704,FeCrSi,Half Heusler,18,C1b,5.3284,1.0043,-0.231,0.00,0.0000,0.00
940,NiFeSn,Half Heusler,22,C1b,5.8500,1.0017,0.017,74.94,3.4914,645.83
1023,NiNiP,Half Heusler,25,triclinic,0.0000,-0.4451,NaN,NaN,0.0000,NaN
851,RhTiP,Half Heusler,18,C1b,5.7366,1.0010,-1.075,0.00,0.0000,0.00
692,FeVAl,Half Heusler,16,C1b,5.6000,1.0000,0.108,15.87,0.8768,185.21


In [37]:
half_heusler_df['struct type'].value_counts()

struct type
C1b           371
triclinic      72
tetragonal      6
Name: count, dtype: int64

In [38]:
half_heusler_df[half_heusler_df['struct type']=='tetragonal']

,formula,heusler type,num_electron,struct type,latt const,tetragonality,e_form,pol fermi,mu_b,mu_b saturation
581,CrTiP,Half Heusler,15,tetragonal,5.24,1.2729,-0.226,15.57,2.5301,512.48
582,CrTiAs,Half Heusler,15,tetragonal,5.52,1.2065,-0.009,100.00,3.0000,548.40
621,CrNiGa,Half Heusler,19,tetragonal,5.11,1.4227,-0.007,32.83,3.1048,606.71
926,NiMnAl,Half Heusler,20,tetragonal,5.56,1.0360,-0.040,69.47,3.1633,659.02
942,NiFeAs,Half Heusler,23,tetragonal,5.54,1.0289,-0.025,77.44,3.6740,779.06
943,NiFeSb,Half Heusler,23,tetragonal,5.91,0.9712,0.018,84.24,3.3752,624.51


In [39]:
half_heusler_df=half_heusler_df[half_heusler_df['struct type']=='C1b']

In [40]:
half_heusler_df.shape

(371, 10)

In [41]:
from pymatgen.core import Composition

def has_three_unique_elements(formula):
    return len(Composition(formula).elements) == 3

half_heusler_df = half_heusler_df[half_heusler_df["formula"].apply(has_three_unique_elements)]

In [42]:
half_heusler_df.shape

(335, 10)

In [43]:
HH_df=half_heusler_df[["formula"]]

In [44]:
HH_df.head()

,formula
576,CrTiGa
577,CrTiIn
578,CrTiSi
579,CrTiGe
580,CrTiSn


In [45]:
HH_df['choice']=1

/tmp/ipykernel_58/3202361466.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  HH_df['choice']=1


In [46]:
!pip install pymatgen

In [47]:
HH_df.shape


(335, 2)

In [48]:
HH_df = HH_df.reset_index(drop=True)

In [49]:
HH_df.head()

,formula,choice
0,CrTiGa,1
1,CrTiIn,1
2,CrTiSi,1
3,CrTiGe,1
4,CrTiSn,1


In [50]:
HH_df[HH_df['formula'].duplicated()]

,formula,choice


In [51]:
import requests
import pandas as pd

url_2 = "https://aflowlib.org/API/aflux/?$nspecies(3),compound,aflow_prototype_label_relax,$paging(1,10000)"

data_2 = requests.get(url_2).json()

In [52]:
from pymatgen.core import Composition
def check_abc(formula):
    comp=Composition(formula)
    if sorted(comp.values())==[1,1,1]:
        return True
    else:
        return False


In [53]:
formula = [entry["compound"] for entry in data_2 if "compound" in entry]
print(formula[10])
print(check_abc(formula[10]))

Cu1Ta1Zn1
True


In [54]:
neg=[]
for m in data_2:
    if(check_abc(m['compound']) and m['aflow_prototype_label_relax']!="ABC_cF12_216_b_c_a"):
        neg.append(m['compound'])

In [55]:
len(neg)
neg=set(neg)


In [56]:
len(neg)

877

In [57]:
df_2=pd.DataFrame({"formula":list(neg)})

In [58]:
df_2.head()

,formula
0,Ir1Mo1V1
1,Br1Se1Si1
2,Ba1Sb1Tl1
3,Ag1B1Li1
4,Sb1Sn1Tl1


In [59]:
df_2["choice"]=0

In [60]:
df_2.shape

(877, 2)

In [61]:
df_final=pd.concat([HH_df,df_2])

In [62]:
df_final['choice'].value_counts()

choice
0    877
1    335
Name: count, dtype: int64

In [63]:
duplicates = df_final[df_final['formula'].duplicated(keep=False)]

print(duplicates)

Empty DataFrame
Columns: [formula, choice]
Index: []


In [64]:
df_final.to_csv('check_hh.csv', index=False)